# EduPredict-XAI
## Step 12 — Data Cleaning and Quality Preparation

**Research Topic:** Student Performance Prediction Using Explainable AI  
**Student:** Payal Pramod Pawar  
**Program:** MCA Sem 3  
**Guide:** Mrs. Shaesta Mujawar  

### Objective

The objective of this notebook is to clean and prepare the student performance datasets for further feature engineering and machine learning experiments.

The cleaning process includes:

- Missing value handling
- Duplicate record analysis
- Data type verification
- Invalid value detection
- Categorical value consistency
- Outlier investigation
- Saving cleaned datasets separately from the original datasets

The original raw datasets will not be modified.

In [3]:
# Import libraries
import pandas as pd
import numpy as np
from pathlib import Path

print("Libraries imported successfully.")

Libraries imported successfully.


In [4]:
# Set project paths
PROJECT_ROOT = Path.cwd().parent

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed"

# Create processed folder if it does not exist
PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)

print("Project root:")
print(PROJECT_ROOT)

print("\nRaw data folder:")
print(RAW_DATA_PATH)

print("\nProcessed data folder:")
print(PROCESSED_DATA_PATH)

Project root:
d:\MCA Sem 3\RP\EduPredict-XAI

Raw data folder:
d:\MCA Sem 3\RP\EduPredict-XAI\data\raw

Processed data folder:
d:\MCA Sem 3\RP\EduPredict-XAI\data\processed


In [5]:
# Load the original dataset
main_path = RAW_DATA_PATH / "main_student_performance.csv"

main_df = pd.read_csv(main_path)

print("Dataset loaded successfully.")
print("Shape:", main_df.shape)

display(main_df.head())

Dataset loaded successfully.
Shape: (1000, 12)


,student_id,gender,study_time_hours,attendance_percent,sleep_hours,parental_education,internet_access,extracurricular_activities,part_time_job,previous_grade,final_exam_score,final_grade
0,1,Male,4.0,98.0,6.5,Bachelors,Yes,Yes,No,76.9,100.0,A
1,2,Female,6.3,100.0,5.7,High School,Yes,Yes,Yes,75.5,100.0,A
2,3,Male,4.9,85.3,7.9,Bachelors,Yes,No,Yes,88.5,97.3,A
3,4,Male,2.6,77.5,8.0,NaN,Yes,Yes,No,85.1,83.8,B
4,5,Male,2.2,89.6,4.6,Bachelors,Yes,No,Yes,61.8,68.3,D


In [6]:
# Create a backup copy
df = main_df.copy()

print("Working copy created.")
print("Original shape:", main_df.shape)
print("Working copy shape:", df.shape)

Working copy created.
Original shape: (1000, 12)
Working copy shape: (1000, 12)


In [7]:
# Check the dataset before cleaning
print("Dataset Shape:")
print(df.shape)

print("\nData Types:")
display(df.dtypes)

print("\nMissing Values:")
display(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

Dataset Shape:
(1000, 12)

Data Types:


student_id                      int64
gender                         object
study_time_hours              float64
attendance_percent            float64
sleep_hours                   float64
parental_education             object
internet_access                object
extracurricular_activities     object
part_time_job                  object
previous_grade                float64
final_exam_score              float64
final_grade                    object
dtype: object


Missing Values:


student_id                      0
gender                          0
study_time_hours                0
attendance_percent              0
sleep_hours                     0
parental_education            102
internet_access                 0
extracurricular_activities      0
part_time_job                   0
previous_grade                  0
final_exam_score                0
final_grade                     0
dtype: int64


Duplicate Rows:
0


In [8]:
# Standardize column names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

print("Updated column names:")
print(df.columns.tolist())

Updated column names:
['student_id', 'gender', 'study_time_hours', 'attendance_percent', 'sleep_hours', 'parental_education', 'internet_access', 'extracurricular_activities', 'part_time_job', 'previous_grade', 'final_exam_score', 'final_grade']


In [9]:
# Set target
TARGET = "final_exam_score"

print("Target variable:", TARGET)
print("Target exists:", TARGET in df.columns)

Target variable: final_exam_score
Target exists: True


In [10]:
# Check data types
print("Numerical columns:")

numeric_columns = df.select_dtypes(
    include=np.number
).columns.tolist()

for col in numeric_columns:
    print("→", col)

print("\nCategorical columns:")

categorical_columns = df.select_dtypes(
    exclude=np.number
).columns.tolist()

for col in categorical_columns:
    print("→", col)

Numerical columns:
→ student_id
→ study_time_hours
→ attendance_percent
→ sleep_hours
→ previous_grade
→ final_exam_score

Categorical columns:
→ gender
→ parental_education
→ internet_access
→ extracurricular_activities
→ part_time_job
→ final_grade


In [13]:
# Missing value analysis
missing_count = df.isnull().sum()

missing_percentage = (
    df.isnull().sum() / len(df) * 100
)

missing_report = pd.DataFrame({
    "Missing_Count": missing_count,
    "Missing_Percentage": missing_percentage
})

missing_report = missing_report.sort_values(
    by="Missing_Percentage",
    ascending=False
)

display(missing_report)

,Missing_Count,Missing_Percentage
parental_education,102,10.2
student_id,0,0.0
study_time_hours,0,0.0
gender,0,0.0
attendance_percent,0,0.0
sleep_hours,0,0.0
internet_access,0,0.0
extracurricular_activities,0,0.0
part_time_job,0,0.0
previous_grade,0,0.0


In [14]:
# Handle missing numerical values
numeric_features = [
    col for col in df.select_dtypes(include=np.number).columns
    if col != TARGET
]

print("Numerical features:")
print(numeric_features)

for col in numeric_features:
    if df[col].isnull().sum() > 0:
        median_value = df[col].median()
        df[col] = df[col].fillna(median_value)

print("Missing numerical values handled.")

Numerical features:
['student_id', 'study_time_hours', 'attendance_percent', 'sleep_hours', 'previous_grade']
Missing numerical values handled.


In [16]:
# Handle missing categorical values
categorical_features = df.select_dtypes(
    exclude=np.number
).columns.tolist()

for col in categorical_features:
    if df[col].isnull().sum() > 0:
        mode_value = df[col].mode()[0]
        df[col] = df[col].fillna(mode_value)

print("Missing categorical values handled.")

Missing categorical values handled.


In [17]:
# Check missing values again
remaining_missing = df.isnull().sum()

print("Remaining missing values:")

display(
    remaining_missing[
        remaining_missing > 0
    ]
)

Remaining missing values:


Series([], dtype: int64)

In [18]:
# Duplicate analysis
duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

Duplicate rows: 0


In [19]:
# Check invalid numerical values
numeric_columns = df.select_dtypes(
    include=np.number
).columns.tolist()

for col in numeric_columns:
    print("=" * 60)
    print(col)
    print("Minimum:", df[col].min())
    print("Maximum:", df[col].max())

student_id
Minimum: 1
Maximum: 1000
study_time_hours
Minimum: 0.5
Maximum: 8.1
attendance_percent
Minimum: 54.8
Maximum: 100.0
sleep_hours
Minimum: 3.2
Maximum: 10.0
previous_grade
Minimum: 31.3
Maximum: 100.0
final_exam_score
Minimum: 46.8
Maximum: 100.0


In [20]:
# Outlier detection using IQR
def detect_outliers_iqr(dataframe, columns):
    results = []

    for col in columns:
        Q1 = dataframe[col].quantile(0.25)
        Q3 = dataframe[col].quantile(0.75)

        IQR = Q3 - Q1

        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        outlier_count = (
            (dataframe[col] < lower_bound) |
            (dataframe[col] > upper_bound)
        ).sum()

        results.append({
            "Feature": col,
            "Q1": Q1,
            "Q3": Q3,
            "IQR": IQR,
            "Lower_Bound": lower_bound,
            "Upper_Bound": upper_bound,
            "Outlier_Count": outlier_count
        })

    return pd.DataFrame(results)

outlier_report = detect_outliers_iqr(
    df,
    numeric_features
)

display(outlier_report)

,Feature,Q1,Q3,IQR,Lower_Bound,Upper_Bound,Outlier_Count
0,student_id,250.75,750.25,499.5,-498.50,1499.50,0
1,study_time_hours,2.60,4.50,1.9,-0.25,7.35,4
2,attendance_percent,78.80,91.90,13.1,59.15,111.55,6
3,sleep_hours,5.90,7.60,1.7,3.35,10.15,2
4,previous_grade,61.00,78.40,17.4,34.90,104.50,3


In [21]:
# Check categorical consistency
categorical_columns = df.select_dtypes(
    exclude=np.number
).columns.tolist()

for col in categorical_columns:
    print("=" * 60)
    print(col)
    print(df[col].value_counts(dropna=False))

gender
gender
Female    510
Male      490
Name: count, dtype: int64
parental_education
parental_education
High School    458
Bachelors      308
Masters        184
PhD             50
Name: count, dtype: int64
internet_access
internet_access
Yes    854
No     146
Name: count, dtype: int64
extracurricular_activities
extracurricular_activities
Yes    572
No     428
Name: count, dtype: int64
part_time_job
part_time_job
No     684
Yes    316
Name: count, dtype: int64
final_grade
final_grade
B    354
A    284
C    261
D     89
F     12
Name: count, dtype: int64


In [22]:
# Check data types again
print("Final data types after cleaning:")

display(df.dtypes)

Final data types after cleaning:


student_id                      int64
gender                         object
study_time_hours              float64
attendance_percent            float64
sleep_hours                   float64
parental_education             object
internet_access                object
extracurricular_activities     object
part_time_job                  object
previous_grade                float64
final_exam_score              float64
final_grade                    object
dtype: object

In [23]:
# Final cleaning report
print("=" * 60)
print("FINAL DATA CLEANING REPORT")
print("=" * 60)

print("\nRows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nMissing values:",
      df.isnull().sum().sum())

print("Duplicate rows:",
      df.duplicated().sum())

print("\nNumerical columns:",
      len(df.select_dtypes(include=np.number).columns))

print("Categorical columns:",
      len(df.select_dtypes(exclude=np.number).columns))

FINAL DATA CLEANING REPORT

Rows: 1000
Columns: 12

Missing values: 0
Duplicate rows: 0

Numerical columns: 6
Categorical columns: 6


In [24]:
# Compare before and after cleaning
print("Original shape:", main_df.shape)
print("Cleaned shape:", df.shape)
print("Rows removed:",
      main_df.shape[0] - df.shape[0])

Original shape: (1000, 12)
Cleaned shape: (1000, 12)
Rows removed: 0


In [25]:
# Save the cleaned dataset
cleaned_path = PROCESSED_DATA_PATH / "main_student_performance_cleaned.csv"

df.to_csv(
    cleaned_path,
    index=False
)

print("Cleaned dataset saved successfully!")
print(cleaned_path)

Cleaned dataset saved successfully!
d:\MCA Sem 3\RP\EduPredict-XAI\data\processed\main_student_performance_cleaned.csv


In [26]:
# Verify the saved file
check_df = pd.read_csv(cleaned_path)

print("Saved dataset shape:", check_df.shape)

display(check_df.head())

Saved dataset shape: (1000, 12)


,student_id,gender,study_time_hours,attendance_percent,sleep_hours,parental_education,internet_access,extracurricular_activities,part_time_job,previous_grade,final_exam_score,final_grade
0,1,Male,4.0,98.0,6.5,Bachelors,Yes,Yes,No,76.9,100.0,A
1,2,Female,6.3,100.0,5.7,High School,Yes,Yes,Yes,75.5,100.0,A
2,3,Male,4.9,85.3,7.9,Bachelors,Yes,No,Yes,88.5,97.3,A
3,4,Male,2.6,77.5,8.0,High School,Yes,Yes,No,85.1,83.8,B
4,5,Male,2.2,89.6,4.6,Bachelors,Yes,No,Yes,61.8,68.3,D


In [27]:
# Setting target variable
print(df.columns.tolist())
TARGET = "final_exam_score"

['student_id', 'gender', 'study_time_hours', 'attendance_percent', 'sleep_hours', 'parental_education', 'internet_access', 'extracurricular_activities', 'part_time_job', 'previous_grade', 'final_exam_score', 'final_grade']
